# Notebook 01 — EDA do Universo B3

**Propósito:** Validar visualmente que os dados estão corretos antes de abrir qualquer arquivo de grafo ou curvatura.

**Critérios de saída (todos devem ser verdadeiros para avançar ao Sprint 2):**
- [ ] `returns_b3.parquet` tem zero NaN no universo elegível nas datas de rebalanceamento
- [ ] `universe_b3.csv` cobre pelo menos 80 ativos em qualquer data após 2016
- [ ] Todos os testes de `test_lookahead.py` passam ✅
- [ ] Retorno médio dos ativos está entre -0.1%/dia e +0.3%/dia (sanidade)
- [ ] Volatilidade média está entre 1%/dia e 3%/dia (sanidade)

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

from src.data.loader import DataLoader
from src.data.cleaner import DataCleaner
from src.data.universe import UniverseBuilder
from src.data.config import load_config

# Estilo
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_palette('deep')

cfg = load_config()
print('Configuração carregada ✓')
print(f"Período: {cfg['universe']['start_date']} → {cfg['universe']['end_date']}")
print(f"Liquidez mínima: R$ {cfg['universe']['min_daily_volume_brl']/1e6:.0f}M/dia")

In [ ]:
# ─── 1. CARREGAR DADOS ────────────────────────────────────────
loader  = DataLoader()
cleaner = DataCleaner()

# Carrega preços brutos
prices_raw = loader.load_b3_prices()
print(f'\nPreços brutos: {prices_raw.shape[1]} ativos × {len(prices_raw)} dias')
print(f'Período: {prices_raw.index[0].date()} → {prices_raw.index[-1].date()}')
print(f'NaN total: {prices_raw.isna().sum().sum():,} ({prices_raw.isna().mean().mean():.1%})')

In [ ]:
# ─── 2. LIMPEZA E RETORNOS ───────────────────────────────────
returns = cleaner.prices_to_log_returns(prices_raw)

print(f'\nRetornos limpos: {returns.shape[1]} ativos × {len(returns)} dias')
print(f'Fill rate médio: {returns.notna().mean().mean():.1%}')
print(f'Retorno médio: {returns.mean().mean():.4%}/dia')
print(f'Volatilidade média: {returns.std().mean():.4%}/dia')

In [ ]:
# ─── 3. UNIVERSO ELEGÍVEL ────────────────────────────────────
builder = UniverseBuilder(prices_raw)

# Calcular universo em diferentes datas
test_dates = pd.bdate_range('2015-01-01', '2024-12-31', freq='QS')
universe_sizes = []

for date in test_dates:
    eligible = builder.get_eligible(date)
    universe_sizes.append({'date': date, 'n_eligible': len(eligible)})

universe_ts = pd.DataFrame(universe_sizes).set_index('date')

# Plot
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(universe_ts.index, universe_ts['n_eligible'],
                alpha=0.3, color='steelblue')
ax.plot(universe_ts.index, universe_ts['n_eligible'],
        color='steelblue', linewidth=2)
ax.axhline(y=80, color='red', linestyle='--', alpha=0.5, label='Mínimo desejado (80)')
ax.set_title('Tamanho do Universo Elegível ao Longo do Tempo', fontsize=13, pad=15)
ax.set_xlabel('Data')
ax.set_ylabel('Número de Ativos')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/pre_relatorio/figures/01_universe_size.png', dpi=150)
plt.show()

print(f'\nMédio de ativos elegíveis: {universe_ts.n_eligible.mean():.0f}')
print(f'Mínimo: {universe_ts.n_eligible.min():.0f}')
print(f'Máximo: {universe_ts.n_eligible.max():.0f}')

In [ ]:
# ─── 4. ANÁLISE DE RETORNOS POR ATIVO ───────────────────────
# Estatísticas descritivas
stats = pd.DataFrame({
    'ret_medio_anual':  returns.mean() * 252,
    'vol_anual':        returns.std() * np.sqrt(252),
    'sharpe_proxy':     returns.mean() / returns.std() * np.sqrt(252),
    'skewness':         returns.skew(),
    'kurtosis':         returns.kurt(),
    'pct_nan':          returns.isna().mean(),
}).sort_values('sharpe_proxy', ascending=False)

print('Top 10 ativos por Sharpe proxy:')
print(stats.head(10).round(3).to_string())
print('\nBottom 10:')
print(stats.tail(10).round(3).to_string())

In [ ]:
# ─── 5. DISTRIBUIÇÃO DE RETORNOS vs NORMAL ──────────────────
from scipy import stats as scipy_stats

# Empilhar todos os retornos
all_returns = returns.values.flatten()
all_returns = all_returns[~np.isnan(all_returns)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
ax = axes[0]
ax.hist(all_returns, bins=200, density=True, alpha=0.6,
        color='steelblue', label='Retornos observados')
x = np.linspace(-0.15, 0.15, 1000)
mu, sigma = all_returns.mean(), all_returns.std()
ax.plot(x, scipy_stats.norm.pdf(x, mu, sigma),
        'r-', linewidth=2, label='Normal teórica')
ax.set_xlim(-0.15, 0.15)
ax.set_title('Distribuição de Retornos vs Normal\n(Fat Tails esperado)', fontsize=12)
ax.set_xlabel('Retorno diário')
ax.legend()

# QQ-Plot
ax = axes[1]
sample = np.random.choice(all_returns, 5000, replace=False)
(osm, osr), (slope, intercept, r) = scipy_stats.probplot(sample)
ax.scatter(osm, osr, alpha=0.3, s=5, color='steelblue', label='Dados')
ax.plot(osm, slope*np.array(osm)+intercept, 'r-', linewidth=2, label='Normal')
ax.set_title(f'QQ-Plot vs Normal\n(Desvios nas caudas = fat tails)', fontsize=12)
ax.set_xlabel('Quantis teóricos')
ax.set_ylabel('Quantis observados')
ax.legend()

kurtosis_excess = scipy_stats.kurtosis(all_returns)
print(f'Curtose em excesso: {kurtosis_excess:.2f} (normal=0, esperado >2 para ações)')

plt.tight_layout()
plt.savefig('../reports/pre_relatorio/figures/02_return_distribution.png', dpi=150)
plt.show()

In [ ]:
# ─── 6. HEATMAP DE MISSING DATA ──────────────────────────────
# Visualiza onde estão os NaN — identifica suspenções e delistagens

# Amostragem de ativos e datas para visualização
sample_tickers = returns.columns[:40]  # primeiros 40
sample_dates   = returns.index[::10]   # a cada 10 dias

is_valid = returns.loc[sample_dates, sample_tickers].notna().astype(int)

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(
    is_valid.T,
    cmap='RdYlGn',
    vmin=0, vmax=1,
    cbar=False,
    ax=ax,
    xticklabels=False,
    yticklabels=True,
)
ax.set_title('Cobertura de Dados por Ativo\n(Verde = dados presentes, Vermelho = NaN)', fontsize=12)
ax.set_xlabel('Tempo →')
plt.tight_layout()
plt.savefig('../reports/pre_relatorio/figures/03_data_coverage.png', dpi=150)
plt.show()

In [ ]:
# ─── 7. SALVAR DADOS PROCESSADOS ─────────────────────────────
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(exist_ok=True)

# Retornos limpos
returns.to_parquet(PROCESSED / 'returns_b3.parquet')
print(f'✓ Salvo: returns_b3.parquet ({returns.shape})')

# Universo na última data disponível
last_date = returns.index[-1]
final_universe = builder.get_eligible(last_date)
pd.Series(final_universe, name='ticker').to_csv(
    PROCESSED / 'universe_b3.csv', index=False
)
print(f'✓ Salvo: universe_b3.csv ({len(final_universe)} ativos)')

# Estatísticas do universo
stats.to_parquet(PROCESSED / 'asset_stats.parquet')
print(f'✓ Salvo: asset_stats.parquet')

In [ ]:
# ─── 8. CHECKLIST FINAL ──────────────────────────────────────
print('\n' + '='*60)
print('CHECKLIST DE SAÍDA DO SPRINT 1')
print('='*60)

checks = {
    'returns_b3.parquet existe':         (PROCESSED / 'returns_b3.parquet').exists(),
    'universe_b3.csv existe':            (PROCESSED / 'universe_b3.csv').exists(),
    'Fill rate > 70%':                   returns.notna().mean().mean() > 0.70,
    'Universo > 80 ativos em 2020':      universe_ts.loc['2020':, 'n_eligible'].min() > 80,
    'Ret médio razoável (-0.1% a 0.3%)': -0.001 < returns.mean().mean() < 0.003,
    'Vol média razoável (1% a 3%)':      0.01 < returns.std().mean() < 0.03,
    'Sem preços negativos':              (prices_raw.dropna() > 0).all().all(),
}

all_pass = True
for check, result in checks.items():
    status = '✅' if result else '❌'
    if not result:
        all_pass = False
    print(f'{status} {check}')

print('\n' + '─'*60)
if all_pass:
    print('✅ SPRINT 1 CONCLUÍDO — Autorizado a abrir src/graph/')
else:
    print('❌ SPRINT 1 INCOMPLETO — Corrija os itens acima antes de avançar')
print('─'*60)

print('\n⚠ Lembre-se de rodar os testes antes de avançar:')
print('  cd .. && python -m pytest tests/test_lookahead.py -v')